# CORDEX AFR-22 daily rsds subset memory failure

This notebook reproduces a CDS WPS workflow that failed because of memory use while subsetting daily CORDEX `rsds` data for 1986–1995.

The request selects every month and every year in the time range, so `time_components` does not reduce the 10-year daily time series. The result is deliberately **not** opened with `resp.datasets()` so that client-side loading does not add another source of memory use.


## Original WPS workflow

The JSON payload below is copied from the failing `wps:ComplexData` request.


In [1]:
request = {
    "inputs": {
        "rsds": [
            "c3s-cordex.output.AFR-22.GERICS.MOHC-HadGEM2-ES.historical."
            "r1i1p1.GERICS-REMO2015.v1.day.rsds.v20201015"
        ]
    },
    "steps": {
        "subset_rsds_1": {
            "run": "subset",
            "in": {
                "collection": "inputs/rsds",
                "area": "-20.0,2.0,17.0,22.0",
                "time_components": (
                    "month:jan,feb,mar,apr,may,jun,jul,aug,sep,oct,nov,dec|"
                    "year:1986,1987,1988,1989,1990,1991,1992,1993,1994,1995"
                ),
                "time": "1986/1995",
            },
        }
    },
    "outputs": {"output": "subset_rsds_1/output"},
    "doc": "workflow",
}

request


{'inputs': {'rsds': ['c3s-cordex.output.AFR-22.GERICS.MOHC-HadGEM2-ES.historical.r1i1p1.GERICS-REMO2015.v1.day.rsds.v20201015']},
 'steps': {'subset_rsds_1': {'run': 'subset',
   'in': {'collection': 'inputs/rsds',
    'area': '-20.0,2.0,17.0,22.0',
    'time_components': 'month:jan,feb,mar,apr,may,jun,jul,aug,sep,oct,nov,dec|year:1986,1987,1988,1989,1990,1991,1992,1993,1994,1995',
    'time': '1986/1995'}}},
 'outputs': {'output': 'subset_rsds_1/output'},
 'doc': 'workflow'}

## Build the equivalent Rooki workflow

Importing Rooki contacts the configured WPS service. Change `ROOK_URL` if the reproduction should run against another deployment.


In [2]:
import json
import os

os.environ["ROOK_URL"] = "http://rook.dkrz.de/wps"

from rooki import operators as ops


In [3]:
rsds = ops.Input("rsds", request["inputs"]["rsds"])
subset = ops.Subset(
    rsds,
    area=request["steps"]["subset_rsds_1"]["in"]["area"],
    time=request["steps"]["subset_rsds_1"]["in"]["time"],
    time_components=request["steps"]["subset_rsds_1"]["in"]["time_components"],
)

serialized_request = json.loads(subset._serialise())
serialized_request


{'inputs': {'rsds': ['c3s-cordex.output.AFR-22.GERICS.MOHC-HadGEM2-ES.historical.r1i1p1.GERICS-REMO2015.v1.day.rsds.v20201015']},
 'steps': {'subset_rsds_1': {'run': 'subset',
   'in': {'collection': 'inputs/rsds',
    'area': '-20.0,2.0,17.0,22.0',
    'time': '1986/1995',
    'time_components': 'month:jan,feb,mar,apr,may,jun,jul,aug,sep,oct,nov,dec|year:1986,1987,1988,1989,1990,1991,1992,1993,1994,1995'}}},
 'outputs': {'output': 'subset_rsds_1/output'},
 'doc': 'workflow'}

## Observed failure

The workflow failed because it exceeded the memory available to the server-side job. This identifies the terminating condition as a memory-allocation failure; it does not establish which subset operation or intermediate allocation caused the peak.

A useful comparison is to omit the logically redundant `time_components` argument while keeping the same `time` and `area`, and then compare peak resident memory. Testing shorter time ranges can show how peak memory scales with the number of input files and timesteps.


## Reproduce the failure

The next cell submits the full request and may exceed the Rook server job's memory allocation. Run it only against the deployment being tested.


In [4]:
from time import perf_counter

started_at = perf_counter()
resp = subset.orchestrate()
elapsed_seconds = perf_counter() - started_at

print(f"Orchestration time: {elapsed_seconds:.1f} seconds")
resp.ok, resp.status


Orchestration time: 15.6 seconds


(True, 'ProcessSucceeded')

## Inspect the response without loading data

If the workflow succeeds, read the total output size and URLs from the returned Metalink document without downloading or opening the NetCDF result. If it fails, displaying `resp` preserves the response details for diagnosis.


In [5]:
resp


Metalink URL: http://rook7.cloud.dkrz.de:80/outputs/rook/a601b0a6-9d4d-11f1-bd5e-fa163eb671ca/input.meta4, num files: 2

In [6]:
if resp.ok:
    print(
        f"Total output size from Metalink: {resp.size:,} bytes "
        f"({resp.size_in_mb:.2f} MiB / {resp.size_in_gb:.3f} GiB)"
    )
    print("\nOutput URLs (not downloaded):")
    for url in resp.download_urls():
        print(url)


Total output size from Metalink: 235,032,976 bytes (224.14 MiB / 0.219 GiB)

Output URLs (not downloaded):
http://rook7.cloud.dkrz.de:80/outputs/rook/ad968b0c-9d4d-11f1-bd13-fa163eb671ca/rsds_AFR-22_MOHC-HadGEM2-ES_historical_r1i1p1_GERICS-REMO2015_v1_day_19860101-19901230.nc
http://rook7.cloud.dkrz.de:80/outputs/rook/ad969e26-9d4d-11f1-bd13-fa163eb671ca/rsds_AFR-22_MOHC-HadGEM2-ES_historical_r1i1p1_GERICS-REMO2015_v1_day_19910101-19951230.nc
